In [1]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)
n_rows = 500

# 1. Synthetic Messy Data Generation
data = {
    'Customer_ID': [f'CUST-{1000 + i}' for i in range(n_rows)],
    'Age': np.random.choice([22, 35, 45, 29, 50, np.nan, 'thirty'], size=n_rows),
    'Gender': np.random.choice(['Male', 'M', 'male', 'Female', 'F', 'female', None], size=n_rows),
    'Annual_Income': np.random.choice([45000, 62000, 85000, 120000, np.nan, 9999999], size=n_rows),
    'Join_Date': np.random.choice(['2023-01-15', '15/01/2023', '2023.01.15', 'invalid_date', np.nan], size=n_rows),
    'Total_Spend': np.random.choice([150.5, 300.0, 1200.75, np.nan, -50.0], size=n_rows)
}

df_messy = pd.DataFrame(data)

# Inject duplicate rows deliberately
duplicates = df_messy.iloc[:20].copy()
df_messy = pd.concat([df_messy, duplicates], ignore_index=True)

# Save messy dataset to CSV
df_messy.to_csv('messy_dataset.csv', index=False)

# 2. Generate Initial Data Quality Report
print("=== INITIAL DATA QUALITY REPORT ===")
print(f"Total Rows: {df_messy.shape[0]}")
print(f"Total Columns: {df_messy.shape[1]}")
print(f"\n--- Duplicate Rows Count: {df_messy.duplicated().sum()} ---")
print("\n--- Missing Values Count Per Column ---")
print(df_messy.isnull().sum())
print("\n--- Column Data Types ---")
print(df_messy.dtypes)
print("\n--- First 10 Rows Preview ---")
df_messy.head(10)

=== INITIAL DATA QUALITY REPORT ===
Total Rows: 520
Total Columns: 6

--- Duplicate Rows Count: 20 ---

--- Missing Values Count Per Column ---
Customer_ID        0
Age                0
Gender            70
Annual_Income     87
Join_Date          0
Total_Spend      116
dtype: int64

--- Column Data Types ---
Customer_ID          str
Age                  str
Gender               str
Annual_Income    float64
Join_Date            str
Total_Spend      float64
dtype: object

--- First 10 Rows Preview ---


,Customer_ID,Age,Gender,Annual_Income,Join_Date,Total_Spend
0,CUST-1000,thirty,NaN,62000.0,15/01/2023,300.00
1,CUST-1001,29,female,85000.0,2023.01.15,NaN
2,CUST-1002,50,male,45000.0,invalid_date,150.50
3,CUST-1003,thirty,Female,NaN,invalid_date,1200.75
4,CUST-1004,45,F,NaN,nan,-50.00
5,CUST-1005,50,Male,120000.0,nan,1200.75
6,CUST-1006,50,F,62000.0,2023-01-15,1200.75
7,CUST-1007,thirty,Female,45000.0,15/01/2023,150.50
8,CUST-1008,35,Female,62000.0,nan,150.50
9,CUST-1009,45,Female,45000.0,invalid_date,150.50


In [2]:
# Create a copy for cleaning
df_clean = df_messy.copy()

# 1. Drop Duplicate Rows
df_clean = df_clean.drop_duplicates()

# 2. Fix 'Age': Force non-numeric strings ('thirty') to NaN, convert to float, impute missing with median
df_clean['Age'] = pd.to_numeric(df_clean['Age'], errors='coerce')
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median()).astype(int)

# 3. Standardize 'Gender': Map all variants to 'Male' or 'Female', impute missing with mode
gender_map = {
    'Male': 'Male', 'M': 'Male', 'male': 'Male',
    'Female': 'Female', 'F': 'Female', 'female': 'Female'
}
df_clean['Gender'] = df_clean['Gender'].map(gender_map)
df_clean['Gender'] = df_clean['Gender'].fillna(df_clean['Gender'].mode()[0])

# 4. Fix 'Annual_Income': Replace 9999999 outlier with NaN, impute missing with median
df_clean['Annual_Income'] = df_clean['Annual_Income'].replace(9999999, np.nan)
df_clean['Annual_Income'] = df_clean['Annual_Income'].fillna(df_clean['Annual_Income'].median())

# 5. Fix 'Total_Spend': Set negative spend (-50.0) to NaN, impute missing with median
df_clean['Total_Spend'] = df_clean['Total_Spend'].apply(lambda x: np.nan if x < 0 else x)
df_clean['Total_Spend'] = df_clean['Total_Spend'].fillna(df_clean['Total_Spend'].median())

# 6. Standardize 'Join_Date': Parse mixed date formats to datetime, forward-fill missing/invalid dates
df_clean['Join_Date'] = pd.to_datetime(df_clean['Join_Date'], errors='coerce', format='mixed')
df_clean['Join_Date'] = df_clean['Join_Date'].ffill().bfill()

# 7. Save Cleaned Dataset to CSV
df_clean.to_csv('cleaned_dataset.csv', index=False)

print("Data cleaning pipeline executed successfully!")
print(f"Cleaned Dataset Shape: {df_clean.shape}")
df_clean.head()

Data cleaning pipeline executed successfully!
Cleaned Dataset Shape: (500, 6)


,Customer_ID,Age,Gender,Annual_Income,Join_Date,Total_Spend
0,CUST-1000,35,Female,62000.0,2023-01-15,300.00
1,CUST-1001,29,Female,85000.0,2023-01-15,300.00
2,CUST-1002,50,Male,45000.0,2023-01-15,150.50
3,CUST-1003,35,Female,62000.0,2023-01-15,1200.75
4,CUST-1004,45,Female,62000.0,2023-01-15,300.00


In [3]:
# Compare Data Quality Before and After Cleaning
summary_data = {
    'Metric': [
        'Total Rows', 
        'Duplicate Rows', 
        'Age Nulls', 
        'Gender Nulls', 
        'Annual_Income Nulls', 
        'Total_Spend Nulls', 
        'Invalid/Null Dates'
    ],
    'Before Cleaning': [
        df_messy.shape[0],
        df_messy.duplicated().sum(),
        pd.to_numeric(df_messy['Age'], errors='coerce').isnull().sum(),
        df_messy['Gender'].isnull().sum(),
        df_messy['Annual_Income'].isnull().sum(),
        df_messy['Total_Spend'].isnull().sum(),
        pd.to_datetime(df_messy['Join_Date'], errors='coerce').isnull().sum()
    ],
    'After Cleaning': [
        df_clean.shape[0],
        df_clean.duplicated().sum(),
        df_clean['Age'].isnull().sum(),
        df_clean['Gender'].isnull().sum(),
        df_clean['Annual_Income'].isnull().sum(),
        df_clean['Total_Spend'].isnull().sum(),
        df_clean['Join_Date'].isnull().sum()
    ]
}

df_summary = pd.DataFrame(summary_data)
print("=== BEFORE VS. AFTER DATA QUALITY REPORT ===")
df_summary

=== BEFORE VS. AFTER DATA QUALITY REPORT ===


C:\Users\saadc\AppData\Local\Temp\ipykernel_22468\2806893698.py:19: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  pd.to_datetime(df_messy['Join_Date'], errors='coerce').isnull().sum()


,Metric,Before Cleaning,After Cleaning
0,Total Rows,520,500
1,Duplicate Rows,20,0
2,Age Nulls,150,0
3,Gender Nulls,70,0
4,Annual_Income Nulls,87,0
5,Total_Spend Nulls,116,0
6,Invalid/Null Dates,409,0
